# Notebook 03 · Image preparation, part 2: boxes, five folds and CT-ICH

Prerequisite: notebook `02_preparo_imagens.ipynb` already run (it writes `tc/preparo/cq500_series.csv` and
`tc/preparo/cq500_series_escolhidas.csv` to the Drive). Runs on a CPU; no GPU is needed.

What this part does, in order:

1. **Final CQ500 boxes:** for each examination with haemorrhage, the hand-drawn boxes on the series used,
   plus those that BHX drew only on copies of that series (matched by slice position in the head).
2. **Split of the patients into five folds** for training and validation (cross-validation), balancing the
   haemorrhage subtypes, with a fixed seed.
3. **CT-ICH (test set):** each haemorrhage outline becomes a box, in the same orientation as the CQ500
   images, with a check figure.

**CT-ICH data use agreement:** the images and the boxes derived from them stay **only on the Drive**; only
counts are kept outside it.

Outside Colab (local test), set `PREPARO_LOCAL` to a folder that mimics `dados/` on the Drive.

**Language:** documentation and comments are in English; identifiers, printed messages and the column
names of the files written here are in Portuguese, exactly as in the run that produced the results.

In [ ]:
# 1. Folders and libraries. In Colab the Drive is used; elsewhere, the folder given in PREPARO_LOCAL.
import ast, csv, io, json, os, random, subprocess, sys, zipfile, collections
import concurrent.futures as cf
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DADOS = Path('/content/drive/MyDrive/ich-benchmark/dados')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pydicom', 'nibabel', 'pylibjpeg', 'pylibjpeg-libjpeg'], check=True)
except ImportError:
    DADOS = Path(os.environ['PREPARO_LOCAL'])

import numpy as np
import pydicom   # with pylibjpeg: some CQ500 examinations are stored as lossless JPEG

BHX, CQ500 = DADOS / 'tc' / 'bhx', DADOS / 'tc' / 'cq500'
CTICH, PREPARO = DADOS / 'tc' / 'ct-ich', DADOS / 'tc' / 'preparo'
SEMENTE = 2026                                    # fixed: the same split in every run
TIPOS = ['Epidural', 'Chronic', 'Subdural', 'Intraventricular', 'Subarachnoid', 'Intraparenchymal']
print(BHX, CQ500, CTICH, PREPARO, sep='\n')

In [ ]:
# 2. Series selected in part 1 and hand-drawn BHX boxes
escolhidas = {l['Name']: l for l in csv.DictReader(open(PREPARO / 'cq500_series_escolhidas.csv'))}
inventario = list(csv.DictReader(open(PREPARO / 'cq500_series.csv')))
bhx = [l for l in csv.DictReader(open(BHX / '3_Extrapolation_to_Selected_Series.csv'))
       if l['labelType'] == 'handDrawn']

caixas_por_corte = collections.defaultdict(list)  # SOPInstanceUID -> [(subtype, x, y, width, height)]
for l in bhx:
    d = ast.literal_eval(l['data'])
    caixas_por_corte[l['SOPInstanceUID']].append((l['labelName'], d['x'], d['y'], d['width'], d['height']))

positivos = sorted((n for n, e in escolhidas.items() if e['grupo'] == 'com sangramento'),
                   key=lambda n: int(n.rsplit('-', 1)[1]))
# Other thick-slice series that BHX also annotated in these examinations (copies and bone versions)
outras_marcadas = collections.defaultdict(list)
for s in inventario:
    if (s['Name'] in positivos and s['caixas_a_mao'] == 'True'
            and s['SeriesInstanceUID'] != escolhidas[s['Name']]['SeriesInstanceUID']
            and 2.5 <= float(s['espessura_mm']) <= 6.0):
        outras_marcadas[s['Name']].append(s)
print(len(positivos), 'exames com sangramento;', len(caixas_por_corte), 'cortes com caixa à mão')
print('Exames com outra série grossa marcada:', {n: [s['pasta'] for s in l] for n, l in outras_marcadas.items()})

In [ ]:
# 3. Final boxes per slice of the series used. Reads the header of each slice (position in the head) of the series used
#    and of the other annotated series; the boxes of a slice of a copy move to the slice of the series used at the
#    same position (difference smaller than half a slice thickness), only if that slice has no box yet.
def zip_do_exame(nome):
    recuperado = CQ500 / 'zips_recuperados' / f'{nome}.zip'
    return recuperado if recuperado.exists() else CQ500 / 'zips' / f'{nome}.zip'

def cortes_da_serie(z, pasta):
    """[(posição z em mm, SOPInstanceUID, tamanho do pixel)] em ordem de posição."""
    lista = []
    for m in z.namelist():
        if m.lower().endswith('.dcm') and m.rsplit('/', 2)[-2] == pasta:
            c = pydicom.dcmread(io.BytesIO(z.read(m)), stop_before_pixels=True)
            lista.append((float(c.ImagePositionPatient[2]), c.SOPInstanceUID, float(c.PixelSpacing[0])))
    return sorted(lista)

def caixas_do_exame(nome):
    caminho = zip_do_exame(nome)
    if not caminho.exists():
        return nome, None, 'zip ausente'
    e = escolhidas[nome]
    with zipfile.ZipFile(caminho) as z:
        usada = cortes_da_serie(z, e['pasta'])
        meia = float(e['espessura_mm']) / 2
        linhas = [{'Name': nome, 'SOPInstanceUID': sop, 'corte': i, 'z_mm': round(zz, 2),
                   'tipo': t, 'x': x, 'y': y, 'largura': w, 'altura': h, 'origem': 'série usada'}
                  for i, (zz, sop, _) in enumerate(usada) for t, x, y, w, h in caixas_por_corte.get(sop, [])]
        com_caixa = {l['corte'] for l in linhas}
        for s in outras_marcadas.get(nome, []):
            for zz, sop, px in cortes_da_serie(z, s['pasta']):
                if sop not in caixas_por_corte:
                    continue
                i = min(range(len(usada)), key=lambda k: abs(usada[k][0] - zz))
                if abs(usada[i][0] - zz) >= meia or i in com_caixa or abs(px - usada[i][2]) > 0.01:
                    continue
                linhas += [{'Name': nome, 'SOPInstanceUID': usada[i][1], 'corte': i, 'z_mm': round(usada[i][0], 2),
                            'tipo': t, 'x': x, 'y': y, 'largura': w, 'altura': h, 'origem': f'cópia {s["pasta"]}'}
                           for t, x, y, w, h in caixas_por_corte[sop]]
                com_caixa.add(i)
    return nome, linhas, ''

with cf.ThreadPoolExecutor(8) as ex:
    resultado = list(ex.map(caixas_do_exame, positivos))
caixas = [l for _, linhas, _ in resultado if linhas for l in linhas]
faltando = [n for n, _, erro in resultado if erro]
with open(PREPARO / 'cq500_caixas.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['Name', 'SOPInstanceUID', 'corte', 'z_mm', 'tipo', 'x', 'y', 'largura',
                                      'altura', 'origem'])
    w.writeheader(); w.writerows(caixas)
de_copia = [l for l in caixas if l['origem'] != 'série usada']
print(len(caixas), 'caixas em', len({(l['Name'], l['corte']) for l in caixas}), 'cortes de',
      len({l['Name'] for l in caixas}), 'exames')
print('Vindas de cópias:', len(de_copia), 'caixas em', len({(l['Name'], l['corte']) for l in de_copia}),
      'cortes', dict(collections.Counter(l['Name'] for l in de_copia)))
print('Caixas por tipo:', dict(collections.Counter(l['tipo'] for l in caixas)))
if faltando:
    print('ATENÇÃO, zips ausentes:', faltando)

In [ ]:
# 4. Split into five folds by patient, balancing haemorrhage subtypes (fixed seed).
#    Starts with the rarest subtype: each examination with it goes to the fold with the fewest examinations of that subtype
#    (ties: fewest examinations with haemorrhage, then random). Normal and equivocal examinations are distributed separately,
#    in rotation after shuffling, so that each fold has the same number of normal examinations.
tipos_do_exame = collections.defaultdict(set)
series_validas = collections.defaultdict(set)
for n in positivos:
    series_validas[n] = {escolhidas[n]['SeriesInstanceUID']} | {s['SeriesInstanceUID'] for s in outras_marcadas.get(n, [])}
nome_da_serie = {uid: n for n, uids in series_validas.items() for uid in uids}
for l in bhx:
    if l['SeriesInstanceUID'] in nome_da_serie:
        tipos_do_exame[nome_da_serie[l['SeriesInstanceUID']]].add(l['labelName'])

sorteio = random.Random(SEMENTE)
parte = {}
por_tipo = {t: [0] * 5 for t in TIPOS}
total = [0] * 5
ordem = positivos[:]; sorteio.shuffle(ordem)
for t in sorted(TIPOS, key=lambda t: sum(t in tipos_do_exame[n] for n in positivos)):
    for n in ordem:
        if n in parte or t not in tipos_do_exame[n]:
            continue
        k = min(range(5), key=lambda k: (por_tipo[t][k], total[k], sorteio.random()))
        parte[n] = k + 1; total[k] += 1
        for t2 in tipos_do_exame[n]:
            por_tipo[t2][k] += 1
for grupo in ('normal', 'duvidoso'):
    nomes = sorted((n for n, e in escolhidas.items() if e['grupo'] == grupo), key=lambda n: int(n.rsplit('-', 1)[1]))
    sorteio.shuffle(nomes)
    for i, n in enumerate(nomes):
        parte[n] = i % 5 + 1

linhas = [{'Name': n, 'grupo': escolhidas[n]['grupo'], 'parte': parte[n],
           'tipos': ';'.join(sorted(tipos_do_exame.get(n, ())))}
          for n in sorted(parte, key=lambda n: int(n.rsplit('-', 1)[1]))]
with open(PREPARO / 'particao_5partes.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=linhas[0].keys()); w.writeheader(); w.writerows(linhas)

tabela = {g: [sum(1 for l in linhas if l['grupo'] == g and l['parte'] == k) for k in range(1, 6)]
          for g in ('com sangramento', 'normal', 'duvidoso')}
tabela.update({t: por_tipo[t] for t in TIPOS})
print(f'{"":<18}' + ''.join(f'parte {k:<3}' for k in range(1, 6)))
for g, v in tabela.items():
    print(f'{g:<18}' + ''.join(f'{x:<9}' for x in v))
sem_tipo = [n for n in positivos if not tipos_do_exame[n]]
assert not sem_tipo, sem_tipo
assert len(set(parte)) == len(parte)        # each patient in a single fold

In [ ]:
# 5. CT-ICH: outline -> box. Each separate region of the mask on a slice becomes one box (connected
#    components). Regions smaller than 10 pixels (~2 mm^2) are discarded and counted.
#    Orientation: the NIfTI volume is brought to the standard orientation (RAS) and flipped to that of the CQ500 DICOM images
#    (rows from anterior to posterior, columns from the patient's right to left).
import nibabel as nib
from scipy import ndimage

MIN_PIXELS = 10

def para_dicom(volume):
    """(x, y, z) em RAS → (z, linha, coluna) como no DICOM."""
    return np.transpose(volume[::-1, ::-1, :], (2, 1, 0))

def abrir(caminho):
    img = nib.as_closest_canonical(nib.load(caminho))
    return para_dicom(img.get_fdata(dtype=np.float32)), [float(x) for x in img.header.get_zooms()[:3]]

volumes = sorted(p for p in CTICH.rglob('*.nii*') if 'mask' not in str(p).lower())
mascaras = {p.name: p for p in CTICH.rglob('*.nii*') if 'mask' in str(p).lower()}

def caixas_ctich(v):
    m, espac = abrir(mascaras[v.name])
    linhas, descartadas = [], 0
    for k in range(m.shape[0]):
        rotulos, n = ndimage.label(m[k] > 0)
        for i, fatia in enumerate(ndimage.find_objects(rotulos), start=1):
            if (rotulos[fatia] == i).sum() < MIN_PIXELS:
                descartadas += 1; continue
            (y0, y1), (x0, x1) = (fatia[0].start, fatia[0].stop), (fatia[1].start, fatia[1].stop)
            linhas.append({'exame': v.name.split('.')[0], 'corte': k, 'tipo': 'Hemorrhage',
                           'x': x0, 'y': y0, 'largura': x1 - x0, 'altura': y1 - y0})
    return v.name, linhas, descartadas, m.shape, espac

with cf.ThreadPoolExecutor(8) as ex:
    res = list(ex.map(caixas_ctich, volumes))
caixas_ct = [l for _, linhas, _, _, _ in res for l in linhas]
with open(CTICH / 'ctich_caixas.csv', 'w', newline='') as f:          # Drive only (data use agreement)
    w = csv.DictWriter(f, fieldnames=['exame', 'corte', 'tipo', 'x', 'y', 'largura', 'altura'])
    w.writeheader(); w.writerows(caixas_ct)
lados = [max(l['largura'], l['altura']) for l in caixas_ct]
resumo_ct = {
    'exames': len(res), 'exames_com_caixa': len({l['exame'] for l in caixas_ct}),
    'cortes': int(sum(s[0] for *_, s, _ in res)),
    'cortes_com_caixa': len({(l['exame'], l['corte']) for l in caixas_ct}),
    'caixas': len(caixas_ct), 'manchas_descartadas_menos_de_10_pixels': int(sum(d for _, _, d, _, _ in res)),
    'lado_maior_da_caixa_pixels_mediana': float(np.median(lados)) if lados else None,
    'tamanhos_de_corte': sorted({tuple(int(x) for x in s[1:]) for *_, s, _ in res}),
    'pixel_mm_faixa': [min(e[0] for *_, e in res), max(e[0] for *_, e in res)],
    'espessura_mm': sorted({round(e[2], 2) for *_, e in res}),
}
(PREPARO / 'ctich_contagens.json').write_text(json.dumps(resumo_ct, indent=2, ensure_ascii=False))
print(json.dumps(resumo_ct, indent=2, ensure_ascii=False))

In [ ]:
# 6. Visual check (kept on the Drive only): a CT-ICH slice next to a CQ500 slice, same orientation,
#    brain window, CT-ICH boxes in yellow. Check: nose up, outlines inside the boxes.
import matplotlib
if 'google.colab' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def janela(hu, nivel=40, largura=80):
    return np.clip((hu - (nivel - largura / 2)) / largura, 0, 1)

if caixas_ct:
    por_corte = collections.Counter((l['exame'], l['corte']) for l in caixas_ct)
    (ex_ct, k_ct), _ = por_corte.most_common(1)[0]
    v = next(p for p in volumes if p.name.split('.')[0] == ex_ct)
    vol, _ = abrir(v); msk, _ = abrir(mascaras[v.name])
    fig, eixos = plt.subplots(1, 2, figsize=(10, 5))
    eixos[0].imshow(janela(vol[k_ct]), cmap='gray')
    eixos[0].contour(msk[k_ct] > 0, levels=[0.5], colors='red', linewidths=0.6)
    for l in caixas_ct:
        if (l['exame'], l['corte']) == (ex_ct, k_ct):
            eixos[0].add_patch(Rectangle((l['x'], l['y']), l['largura'], l['altura'], fill=False, color='yellow'))
    eixos[0].set_title(f'CT-ICH {ex_ct}, corte {k_ct}\ncontorno (vermelho) e caixa (amarelo)', fontsize=10)
    nome_cq = next(n for n in positivos if zip_do_exame(n).exists())
    with zipfile.ZipFile(zip_do_exame(nome_cq)) as z:
        usada = cortes_da_serie(z, escolhidas[nome_cq]['pasta'])
        meio = usada[len(usada) // 2][1]
        arq = next(m for m in z.namelist() if m.lower().endswith('.dcm') and m.rsplit('/', 2)[-2] == escolhidas[nome_cq]['pasta']
                   and pydicom.dcmread(io.BytesIO(z.read(m)), stop_before_pixels=True).SOPInstanceUID == meio)
        d = pydicom.dcmread(io.BytesIO(z.read(arq)))
    eixos[1].imshow(janela(d.pixel_array * float(d.RescaleSlope) + float(d.RescaleIntercept)), cmap='gray')
    eixos[1].set_title(f'{nome_cq}, corte do meio\nreferência de orientação', fontsize=10)
    for e in eixos:
        e.axis('off')
    fig.tight_layout(); fig.savefig(CTICH / 'conferencia_orientacao.png', dpi=120)
    plt.show()

## After running

1. Check cell 3: the boxes taken from copies should be in examinations CQ500-CT-277, 338 and 449.
2. Check the figure of cell 6: CT-ICH must be in the same orientation as CQ500, with each outline inside its
   box.
3. The files `tc/preparo/cq500_caixas.csv`, `tc/preparo/particao_5partes.csv` and
   `tc/preparo/ctich_contagens.json` describe the reference boxes, the folds and the CT-ICH counts. Nothing
   from `tc/ct-ich/` may be shared (CT-ICH data use agreement).